# Adaptive planner-head agent — train all stages & compare SFT vs SFT+RL

End-to-end: **SFT (two-stage curriculum) → offline rollouts → offline GRPO → inference**, then
**compare old SFT against new SFT+RL on all 1000 instructions** (correctness, plan-vs-no-plan
ablation gap, plan-token distribution diff, adapter L2 diff).

Needs a GPU and downloads `Qwen/Qwen2.5-1.5B-Instruct` on first run. Set `SMOKE=True` in the config
cell to dry-run the whole pipeline on a tiny slice first. All addenda (A1–A8) are wired into the
scripts this notebook calls; see `important_design_choices.md`.

In [ ]:
# !pip install -r requirements.txt   # transformers>=4.51, peft, torch, accelerate
import os, json, subprocess, sys, torch

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
SMOKE  = False                     # True -> tiny slice, fast sanity pass
DATA   = 'dataset/sft_flat.jsonl'  # 1000 rows (already annotated with reward_path)

N_TRAIN, N_HELD = (40, 10) if SMOKE else (900, 100)
ROLL_TRAIN      = 40 if SMOKE else 900
GROUP, TEMP     = 8, 1.5
MAXR            = 64

def run(cmd):
    print('>>', ' '.join(cmd)); sys.stdout.flush()
    subprocess.run(cmd, check=True)

print('device =', DEVICE, '| smoke =', SMOKE, '| data =', DATA)

## Stage 1 — SFT (A3 two-stage curriculum: broad → hard)
Logs the held **ablation gap** after each stage (should grow in stage 2) and the A4 probe diversity.

In [ ]:
run([sys.executable, 'train_sft.py', '--data', DATA, '--train', str(N_TRAIN), '--held', str(N_HELD),
     '--curriculum', '--stage1_epochs', '2' if SMOKE else '5', '--stage2_epochs', '1' if SMOKE else '2',
     '--lr', '5e-5', '--lr_min', '8e-8', '--max_resp', str(MAXR), '--out', 'joint_ckpt',
     '--eval_sample', '--device', DEVICE])

## Stage 2a — offline rollouts (generate ONCE)
Samples G=8 hot rollouts/prompt, graded reward via `reward_path`, A2 pre-RL filter →
`rl_rollouts.jsonl` + `pre_rl_filter_report.csv`.

In [ ]:
run([sys.executable, 'rollout_offline.py', '--sft_ckpt', 'joint_ckpt', '--data', DATA,
     '--train', str(ROLL_TRAIN), '--group', str(GROUP), '--temp', str(TEMP), '--max_resp', str(MAXR),
     '--out', 'rl_rollouts.jsonl', '--device', DEVICE])

## Stage 2b — offline GRPO (A1/A1b MaxEnt weighting, A5 long2short, A6 mismatch guard)
Startup prints `>0 trainable backbone tensors` and `logp_mismatch_t0` (~0). Watch `exec_approx_kl`
and `held_reward` move across inner epochs; `p_q_hist` shows the MaxEnt bell at work.

In [ ]:
run([sys.executable, 'train_grpo_offline.py', '--rollouts', 'rl_rollouts.jsonl', '--sft_ckpt', 'joint_ckpt',
     '--data', DATA, '--out', 'rl_ckpt', '--inner_epochs', '2' if SMOKE else '3', '--lr', '1e-4',
     '--clip_eps', '0.2', '--beta_plan', '1.0', '--beta_ce', '0.1', '--gamma', '2.0',
     '--long2short', '--max_resp', str(MAXR), '--device', DEVICE])

## Inference & comparison — old SFT vs new SFT+RL on ALL 1000 instructions
Sampled decoding (temp 0.7) with a sampled plan (deployment mode). Reports correctness, the
plan-vs-no-plan ablation gap, the plan-token distribution diff, and the adapter L2 diff (>0 proves
RL moved the backbone).

In [ ]:
from collections import Counter
from model_joint import JointModel, encode_plan, decode_plan
from checkers import graded_reward_for_row
import compare

rows = [json.loads(l) for l in open(DATA)]
if SMOKE: rows = rows[:20]
print(f'evaluating {len(rows)} instructions per checkpoint')

@torch.no_grad()
def eval_ckpt(ckpt):
    m = JointModel.from_checkpoint('Qwen/Qwen2.5-1.5B-Instruct', ckpt, device=DEVICE, is_trainable=False)
    m.eval(); acc=plan=none=0.0; dist=Counter()
    for r in rows:
        p_ids,p_attn = m.batch_prompts([r['instruction']])
        pl = m.sample_plan(p_ids,p_attn,temp=0.7,sample=True); dist.update(decode_plan(pl[0]))
        g  = m.generate_answer(p_ids,p_attn,pl,sample=True,temp=0.7,max_new_tokens=MAXR)
        acc += graded_reward_for_row(r, m.tok.decode(g[0],skip_special_tokens=True))
        gp = encode_plan(r['plan'], m.plan_max_len).unsqueeze(0).to(m.device)
        a1 = m.generate_answer(p_ids,p_attn,gp,sample=True,temp=0.7,max_new_tokens=MAXR)
        a0 = m.generate_answer(p_ids,p_attn,None,sample=True,temp=0.7,max_new_tokens=MAXR)
        plan += graded_reward_for_row(r, m.tok.decode(a1[0],skip_special_tokens=True))
        none += graded_reward_for_row(r, m.tok.decode(a0[0],skip_special_tokens=True))
    n=len(rows); del m
    return {'acc':acc/n,'ablation_gap':(plan-none)/n,'plan_dist':dict(dist.most_common(10))}

sft = eval_ckpt('joint_ckpt'); print('SFT   ', sft)
rl  = eval_ckpt('rl_ckpt');    print('SFT+RL', rl)
l2  = compare.lora_param_diff('Qwen/Qwen2.5-1.5B-Instruct', 'joint_ckpt', 'rl_ckpt', DEVICE)
print(f'\nacc:  SFT={sft["acc"]:.3f}  SFT+RL={rl["acc"]:.3f}')
print(f'ablation_gap:  SFT={sft["ablation_gap"]:+.3f}  SFT+RL={rl["ablation_gap"]:+.3f}')
print(f'adapter L2 |RL-SFT| = {l2:.5f}  (>0 proves RL != SFT)')

### Acceptance recap
- SFT held `plan_ce` dropped; **ablation_gap positive** (stage2 ≥ stage1 with curriculum).
- GRPO startup asserted **>0 trainable backbone tensors** and **logp_mismatch_t0 ≈ 0** (A6).
- GRPO `p_q_hist` shows the MaxEnt bell; `held_reward` moved across inner epochs.
- **SFT+RL ≠ SFT** under sampling and **adapter L2 diff > 0**.
- Optional: `python compare.py --sft_ckpt joint_ckpt --rl_ckpt rl_ckpt --sample --clr --device cuda`
  adds the A8 claim-level scaling number; `python self_distill.py --rl_ckpt rl_ckpt ...` (A7)
  consolidates RL gains.